# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #1: "Captured Traffic Value" (Finding #9)** — clicks × CPC =
$253.5K captured value, rejecting impressions × CPC ($73.0M) as unsafe.

Methodology question: Where does the CPC figure come from, and is it
matched per-keyword or a portfolio-wide benchmark? The paper correctly
rejects impressions × CPC as unsafe, but clicks × CPC is only as
defensible as the CPC number underneath it. If CPC is a rough external
benchmark rather than matched per-keyword, the $253.5K total inherits
that same uncertainty, just one level removed. I'd ask: does CPC vary
by keyword/intent in this calculation, or is one benchmark applied
across all 341K pages?

**Finding #2: "What Predicts Health?" (ML Appendix)** — Random Forest:
Average Position (43% importance), Impressions (32%), Scroll Depth
(15%) predicting Health Score.

Methodology question: The paper itself discloses this honestly —
"the target itself is partly constructed from some of these inputs,
so importance is descriptive rather than causal" — since Health Score
is literally built from impressions + position + CTR + scroll depth.
My question: given that 3 of the top 4 predicted features are also
literal scoring components of the label, does this finding tell us
anything beyond confirming the scoring formula works as designed? I'd
want to see feature importance recomputed against a target Health
Score doesn't already encode (e.g. future impression change), to know
if there's real predictive signal beyond circularity.

**Framing note:** both questions mirror the same standard I apply to
my own model below — not "this paper did it wrong" (it discloses both
caveats itself), but "here's the next check I'd want before fully
trusting the number."

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/imnotparama/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count", "search_volume"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# BEFORE: naive random split
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.25, random_state=42)
rf_random = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf_random.fit(X_train_r, y_train_r)
random_p50 = precision_at_k(rf_random.predict_proba(X_test_r)[:, 1], y_test_r, 50)

# AFTER: grouped split by client
groups = df["client_id"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

rf_grouped = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf_grouped.fit(X_train_g, y_train_g)
grouped_p50 = precision_at_k(rf_grouped.predict_proba(X_test_g)[:, 1], y_test_g, 50)

comparison = pd.DataFrame({
    "Split": ["Random (before)", "Grouped by client (after)"],
    "Base rate": [y_test_r.mean(), y_test_g.mean()],
    "Precision@50": [random_p50, grouped_p50]
})
print(comparison)

Working dir: /content/flyrank-ml-internship
                       Split  Base rate  Precision@50
0            Random (before)   0.546000          0.92
1  Grouped by client (after)   0.516514          0.58


**Before/after finding:** The gap is large and important. Precision@50
was 0.920 under a naive random split, but drops to 0.580 under a
grouped-by-client split — a 34-point collapse. This confirms the
random split was letting the model partially memorize client-specific
patterns (since a client's pages can appear in both train and test),
which inflated the apparent skill. The grouped number (0.580) is the
honest one: it tests whether the model works on clients it has never
seen, which is the real deployment question. I can explain the gap
entirely by client leakage across the random split — this is not a
mystery, it's the expected signature of an dishonest split.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [2]:
print("Attack checklist:")
print("- Timeline: all 7 features observable at prediction time, none")
print("  derived from trend_pct/trend_direction (the label source).\n")

# Deliberately add the leaky feature to prove the harness catches it
X_leaky = X.copy()
X_leaky["trend_pct_LEAKY"] = df["trend_pct"]

X_train_leak, X_test_leak = X_leaky.iloc[train_idx], X_leaky.iloc[test_idx]
rf_leaky = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf_leaky.fit(X_train_leak, y_train_g)
leaky_p50 = precision_at_k(rf_leaky.predict_proba(X_test_leak)[:, 1], y_test_g, 50)

print(f"WITHOUT leaky feature (honest, grouped): Precision@50 = {grouped_p50:.3f}")
print(f"WITH trend_pct as a feature (deliberately leaky): Precision@50 = {leaky_p50:.3f}")
print("\nThe jump confirms the test harness correctly detects leakage.")
print("trend_pct/trend_direction are excluded from the real model — this was a test only.")

# Base rate check
print(f"\nBase rate (grouped test set): {y_test_g.mean():.3f}")
print("Every Precision@50 number above should be read against this base rate,")
print("not treated as skill on its own.")

Attack checklist:
- Timeline: all 7 features observable at prediction time, none
  derived from trend_pct/trend_direction (the label source).

WITHOUT leaky feature (honest, grouped): Precision@50 = 0.580
WITH trend_pct as a feature (deliberately leaky): Precision@50 = 1.000

The jump confirms the test harness correctly detects leakage.
trend_pct/trend_direction are excluded from the real model — this was a test only.

Base rate (grouped test set): 0.517
Every Precision@50 number above should be read against this base rate,
not treated as skill on its own.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest original claim (from ML-08 notes):** "Random Forest
(0.580) actually underperformed both the baseline rule (0.620) and
Logistic Regression (0.660)."

This claim already used fairly careful language, but rereading it,
here's a stricter rewrite:

**Rewritten:** "On this specific grouped-by-client test split (8 held
-out clients, random_state=42), Random Forest's observed Precision@50
was directionally lower than both the baseline rule and Logistic
Regression. This is a single-split observation, not a stable ranking
of methods — a different random seed or a larger held-out client set
could shift these numbers. It's decision-support evidence that
complexity didn't earn its keep here, not proof that Random Forest is
categorically worse for this task."

The key changes: naming the exact split/seed instead of implying a
general truth, calling it "directionally lower" instead of a flat
statement, and explicitly flagging that a different split could
change the ranking — which the skill file's own base-rate and
gap-reporting guidance pushes toward.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.